# EP 3 - Memoria e stato: sessioni e context provider

In EP 2 il nostro agente ha imparato ad **agire** (tool + MCP). Ma c'e' un problema: a ogni `run` **riparte da zero**. Oggi gli diamo la **memoria**, in tre livelli veri:

1. **Sessione** - ricorda dentro *una* conversazione;
2. **Memoria persistente** - ricorda *tra* le sessioni chi sei (i tuoi gusti);
3. **Compaction** - quando la storia cresce, la **compatta** per non sforare il contesto.

Il protagonista e' il **Fan** del Debate Club (il filo dell'"opinionista" di EP 1/2): difende i film che ama. Come faranno i debater da EP 4, **non ha tool**: argomenta **solo dal dossier** che il Ricercatore ha preparato in EP 2. Qui gli aggiungiamo la memoria - cosi' e' gia' pronto per il Debate Club.

Companion teorico: `teoria.md`. Notebook per **Colab e VS Code**: la cella di setup rileva l'ambiente.

## 1. Setup

Rileviamo l'ambiente, installiamo le dipendenze dove serve, carichiamo la chiave OpenAI.

> In VS Code: ambiente `uv` (`uv add agent-framework-core agent-framework-openai python-dotenv`). Su Colab la cella installa il minimo. Da EP 2 in poi usiamo **solo OpenAI** (memoria e compaction vogliono un modello affidabile).
>
> Promemoria da EP 1: **MAF non carica `.env` da solo**, serve `load_dotenv()`.

In [1]:
import sys, subprocess, os, warnings

warnings.filterwarnings("ignore", message=".*experimental.*")  # l'harness di sez. 5c e' experimental

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "agent-framework-core==1.9.0", "agent-framework-openai==1.8.2", "python-dotenv==1.2.2"],
        check=True,
    )

from dotenv import load_dotenv
load_dotenv()  # MAF non carica .env da solo

if not os.environ.get("OPENAI_API_KEY"):
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

MODEL = "gpt-4o-mini"
print("Ambiente:", "Colab" if IN_COLAB else "locale (VS Code/uv)")

Ambiente: locale (VS Code/uv)


## 2. Il materiale: i dossier del Ricercatore (EP 2)

Il Fan non cerca dati da solo: legge il **dossier** che il Ricercatore ha prodotto in EP 2 (numeri + trama + recensioni con fonti). E' il **contratto tra agenti** che ritroveremo nel Debate Club.

Ne carichiamo **quattro** - un file JSON per film, come **quattro chiamate** distinte del Ricercatore (stesso regista, M. Night Shyamalan, voti dal disastro al capolavoro). In locale stanno in `dossier/`; su Colab la cella li scarica dal repo. Il notebook non tiene piu' i dati inline: **legge i file**, come farebbe con l'output vero del Ricercatore.

In [2]:
import json, urllib.request
from pathlib import Path

# I 4 dossier sono 4 output del Ricercatore di EP 2: una chiamata per film (stesso regista, film diversi).
# In locale stanno in dossier/; su Colab li scarichiamo dal repo pubblico.
FILMS = ["old", "sixth_sense", "split", "last_airbender"]
DOSSIER_DIR = Path("dossier")
RAW_BASE = "https://raw.githubusercontent.com/boosha-ai/boosha-agentic-corso-maf/main/maf/ep03_memoria_stato/dossier"

def carica_dossier(slug: str) -> dict:
    """Carica un dossier: dal file in locale, via download su Colab."""
    f = DOSSIER_DIR / f"{slug}.json"
    testo = f.read_text() if f.exists() else urllib.request.urlopen(f"{RAW_BASE}/{slug}.json").read().decode()
    return json.loads(testo)

DOSSIER = {slug: carica_dossier(slug) for slug in FILMS}

def dossier_str(slug: str) -> str:
    """Rende un dossier come testo compatto da passare al Fan nel contesto."""
    d = DOSSIER[slug]
    righe = [
        f"Titolo: {d['titolo']} ({d['anno']}) - regia di {d['regista']}",
        f"Incasso: {d['incasso']} | Rotten Tomatoes: {d['rotten_tomatoes']}",
        f"Trama: {d['descrizione']}",
        "Recensioni:",
    ]
    for r in d["recensioni"]:
        righe.append(f"  - [{r['sentiment']}] {r['fonte']}: {r['estratto']}")
    return "\n".join(righe)

for slug, d in DOSSIER.items():
    print(f"{slug:15s} {d['titolo']:20s} RT {d['rotten_tomatoes']:>4s}  ({len(d['recensioni'])} recensioni)")

old             Old                  RT  50%  (3 recensioni)
sixth_sense     The Sixth Sense      RT  86%  (6 recensioni)
split           Split                RT  79%  (6 recensioni)
last_airbender  The Last Airbender   RT   5%  (6 recensioni)


## 3. Il problema: il Fan e' smemorato

Creiamo il Fan e gli facciamo commentare un film. Poi, in una **nuova** chiamata, gli chiediamo di quel film. Senza memoria, ogni `run` e' un mondo a se': non sa di cosa stessimo parlando.

In [3]:
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient

FAN = (
    "Sei il Fan del Debate Club: difendi con entusiasmo i film che ami, ma resti onesto. "
    "Argomenta SOLO con i dati del dossier che ti viene dato (incassi, voti, recensioni): "
    "non inventare numeri. Rispondi in italiano, in 2-3 frasi."
)

fan = Agent(client=OpenAIChatClient(model=MODEL), name="Fan", instructions=FAN)

# primo run: commenta un film
r1 = await fan.run(f"Ecco il dossier di un film:\n{dossier_str('old')}\nDifendilo in breve.")
print("Fan:", r1.text.strip())

# secondo run, SENZA memoria: non sa di cosa parlassimo
r2 = await fan.run("E quel film di prima, come si chiamava?")
print("\nFan (nuovo run):", r2.text.strip())
print("\n-> nessuna memoria: ha gia' dimenticato.")

Fan: "Old" di M. Night Shyamalan, nonostante un punteggio di Rotten Tomatoes del 50%, ha incassato ben 90.2 milioni di dollari, attestando il suo successo commerciale. Le recensioni su Metacritic evidenziano come il film riesca a bilanciare elementi di serietà e leggerezza, esplorando temi profondi come il tempo e l'importanza di vivere nel presente. Inoltre, i commenti degli utenti sottolineano come la trama coinvolgente mantenga alta l'attenzione, rendendo "Old" un'opera meritevole di visione.

Fan (nuovo run): Non ho informazioni sui film di cui stai parlando. Se puoi fornirmi il titolo del film o i dati pertinenti, sarò felice di aiutarti a difenderlo!

-> nessuna memoria: ha gia' dimenticato.


## 4. Livello 1 - la sessione

La **sessione** (`agent.create_session()`) tiene insieme i turni di *una* conversazione. La si passa a `run(..., session=...)`: da qui il Fan ricorda cosa ci siamo detti.

In [4]:
ses = fan.create_session()

await fan.run(f"Ecco il dossier di un film:\n{dossier_str('old')}\nDifendilo in breve.", session=ses)
r = await fan.run("E quel film di prima, come si chiamava? E che voto aveva?", session=ses)
print("Fan (in sessione):", r.text.strip())
print("\n-> ricorda: stessa conversazione, stesso filo.")

Fan (in sessione): Il film di prima si chiama "Old" ed è stato diretto da M. Night Shyamalan. Ha un punteggio di Rotten Tomatoes del 50%.

-> ricorda: stessa conversazione, stesso filo.


## 5. Livello 2 - la memoria di *te*, tra le sessioni

La sessione dura una chiacchierata. Ma un buon compagno di cinema dovrebbe ricordarsi **chi sei** anche domani: cosa ami, cosa eviti.

E' il problema del **cold start** dei sistemi di raccomandazione: un utente nuovo non ha storia, quindi non lo si puo' personalizzare. Netflix lo risolve col suo onboarding ("Jumpstart"): al primo accesso ti fa **scegliere qualche titolo** per partire. Facciamo lo stesso: se non conosciamo ancora i tuoi gusti, il Fan **te li chiede**; poi li salva su file e li rilegge alla prossima sessione.

Lo strumento e' il **context provider**: una classe con due momenti - `before_run` (inietta contesto *prima* del modello) e `after_run` (elabora la risposta *dopo*).

In [5]:
from typing import Any
from pathlib import Path
from agent_framework import ContextProvider

PROFILO = Path("profilo_gusti.txt")   # gli appunti del Fan su di te, su file

class ProfiloGusti(ContextProvider):
    """Ricorda i gusti cinematografici dell'utente tra una sessione e l'altra."""

    def __init__(self, path: Path):
        super().__init__("gusti")
        self.path = Path(path)

    def _leggi(self) -> str:
        return self.path.read_text().strip() if self.path.exists() else ""

    async def before_run(self, *, agent, session, context, state: dict[str, Any]) -> None:
        profilo = self._leggi()
        if profilo:
            context.extend_instructions(self.source_id, f"Profilo gusti dell'utente (dai tuoi appunti):\n{profilo}")
        else:
            # cold start: nessuna storia -> chiedi (come Netflix al primo accesso)
            context.extend_instructions(self.source_id, "Non conosci ancora i gusti cinematografici dell'utente: chiediglieli con garbo.")

    async def after_run(self, *, agent, session, context, state: dict[str, Any]) -> None:
        # estrai gusti duraturi dallo scambio e appendili al file (come fa un memory harness, ma a mano)
        scambio = "\n".join(f"{m.role}: {m.text}" for m in context.input_messages if getattr(m, "text", ""))
        risposta = context.response.text if context.response is not None else ""
        estrattore = Agent(
            client=OpenAIChatClient(model=MODEL), name="estrattore",
            instructions=("Estrai i gusti cinematografici DURATURI dell'utente (registi/generi amati o odiati). "
                          "Scrivi in italiano corretto, terza persona singolare, una riga per gusto: "
                          "'- Ama <cosa>' oppure '- Odia <cosa>'. Niente duplicati con quanto gia' noto. "
                          "Se non ce ne sono, rispondi solo 'NIENTE'."),
        )
        out = (await estrattore.run(f"Gia' noto:\n{self._leggi()}\n\nUtente: {scambio}\nAssistente: {risposta}")).text.strip()
        if out and "NIENTE" not in out.upper():
            self.path.write_text((self._leggi() + "\n" + out).strip())

if PROFILO.exists():
    PROFILO.unlink()   # partiamo puliti, come un utente nuovo
print("profilo iniziale:", repr(PROFILO.read_text()) if PROFILO.exists() else "(vuoto)")

profilo iniziale: (vuoto)


### 5a. Prima sessione: il cold start

Profilo vuoto: il Fan non ci conosce e ce lo chiede. Quando gli diciamo i nostri gusti, `after_run` li estrae e li **scrive su file**.

In [6]:
fan_mem = Agent(client=OpenAIChatClient(model=MODEL), name="Fan",
                instructions=FAN, context_providers=[ProfiloGusti(PROFILO)])
ses1 = fan_mem.create_session()

# profilo vuoto -> chiede
print("Fan:", (await fan_mem.run("Ciao!", session=ses1)).text.strip())

# gli diciamo i gusti -> after_run li salva
await fan_mem.run("Amo Christopher Nolan e non sopporto gli horror lenti.", session=ses1)

print("\n--- profilo su file dopo la sessione ---")
print(PROFILO.read_text())

Fan: Ciao! Come posso aiutarti oggi? Quali sono i tuoi gusti cinematografici?

--- profilo su file dopo la sessione ---
- Ama Christopher Nolan  
- Odia gli horror lenti


### 5b. La prova che non e' un pappagallo: stesso film, due spettatori

Se la memoria servisse solo a ripetere quello che le dici, sarebbe un trucco. La prova che **cambia davvero il comportamento**: diamo al Fan **lo stesso dossier** ma **due profili diversi** (due istanze nuove, come due utenti che tornano in giorni diversi). Stesso film, due difese diverse - guidate solo dalla memoria.

In [7]:
# due spettatori con gusti opposti: due FILE di profilo diversi.
# Nessuna sessione in gioco - la memoria persistente vive su disco, non nella conversazione.
prof_a = Path("profilo_a.txt"); prof_a.write_text("- Ama Christopher Nolan\n- Adora i twist finali spiazzanti")
prof_b = Path("profilo_b.txt"); prof_b.write_text("- Detesta i film lenti\n- Odia le storie che si prendono troppo sul serio")

async def difendi_sixth_sense(profilo):
    fan = Agent(client=OpenAIChatClient(model=MODEL), name="Fan",
                instructions=FAN, context_providers=[ProfiloGusti(profilo)])
    r = await fan.run(f"Ecco il dossier:\n{dossier_str('sixth_sense')}\n"
                      "In base ai miei gusti, convincimi a vederlo. Di' su cosa ti basi.")
    return r.text.strip()

print("== Stesso film (The Sixth Sense), due spettatori ==\n")
print("[A: ama Nolan e i twist]\n", await difendi_sixth_sense(prof_a))
print("\n[B: odia i film lenti]\n", await difendi_sixth_sense(prof_b))

== Stesso film (The Sixth Sense), due spettatori ==

[A: ama Nolan e i twist]
 "The Sixth Sense" è un capolavoro di M. Night Shyamalan che, con un incasso di 672.8 milioni di dollari e un punteggio di 86% su Rotten Tomatoes, ha conquistato il pubblico e la critica. La trama intrigante, unita a un finale spiazzante, è descritta come “brillante” e “ben ritmata” da Metacritic, e offre un’adorabile amalgama di inquietudine e sensibilità. Se ami i twist finali sorprendenti, questo film ti lascerà senza fiato!

[B: odia i film lenti]
 "The Sixth Sense" è un thriller psicologico che ha incassato ben $672.8 milioni e ha un punteggio di 86% su Rotten Tomatoes, il che dimostra che ha conquistato un pubblico vasto e critici. Nonostante il suo ritmo delicato, la tensione cresce in modo avvincente e il film sfida le convenzioni con una trama coinvolgente e intelligente, facendo anche riflettere sulle paure più profonde. Se ami storie che sorprendono e colpiscono, questo film potrebbe essere proprio

> **Su che base?** Hai visto: **stesso film, due risposte diverse** - a guidare e' la memoria, non un'etichetta ripetuta. Ma attenzione: qui **non c'e' nessun algoritmo di raccomandazione**. Il Fan **ragiona** con l'LLM sui gusti che gli hai dichiarato; non e' collaborative filtering, non c'e' un modello addestrato sul tuo comportamento ("chi e' come te ha visto..."). Per una vera raccomandazione su larga scala servirebbe un **recommender** vero. L'agente **ricorda e ragiona**, non **impara** un modello di gusti - e' il punto della prossima sezione.

### 5c. Lo stesso, gia' pronto: il memory harness (experimental)

Abbiamo scritto a mano `before_run`/`after_run`. MAF ha gia' un **memory harness** che fa tutto questo (e di piu': estrazione, consolidamento, un `MEMORY.md` con topic) su disco - identico a come funziona la memoria di un assistente vero.

E' **experimental**: lo mostriamo per far vedere "dove va", non come base di produzione.

> Twist onesto: Netflix personalizza **imparando** un modello dai dati (matrix factorization, il Netflix Prize). Il nostro agente **non impara**: i suoi pesi restano congelati. Ricorda soltanto - appunti esterni riletti a ogni run. Ricordare non e' imparare (ne parliamo in `teoria.md`).

In [8]:
import shutil
from base64 import urlsafe_b64decode
from agent_framework import MemoryContextProvider, MemoryFileStore

BASE = Path("mem_harness")
shutil.rmtree(BASE, ignore_errors=True)   # partiamo puliti, come col profilo di sez. 5

store = MemoryFileStore(base_path=BASE, owner_state_key="user_id")
mem = MemoryContextProvider(store=store, consolidation_min_sessions=1)  # scrive subito (default 5)

fan_harness = Agent(client=OpenAIChatClient(model=MODEL), name="Fan",
                    instructions=FAN, context_providers=[mem])
ses_h = fan_harness.create_session()
ses_h.state["user_id"] = "veronica"   # la memoria e' legata a questo utente

await fan_harness.run("Ricordati di me: amo Christopher Nolan e non sopporto gli horror lenti.", session=ses_h)

# DOVE ha scritto: base_path / b64(source_id) / b64(utente) / kind
# I due segmenti di mezzo sono in base64, cosi' qualunque user_id (spazi, accenti, una email)
# diventa un nome di cartella valido e non puo' evadere da base_path.
def decodifica(nome: str) -> str:
    try:
        return urlsafe_b64decode(nome + "=" * (-len(nome) % 4)).decode()
    except Exception:
        return nome

print("albero su disco:\n")
for f in sorted(BASE.rglob("*")):
    rel = f.relative_to(BASE)
    leggibile = decodifica(rel.name) if len(rel.parts) <= 2 else rel.name   # solo i primi 2 sono codificati
    nota = f'   <- "{leggibile}"' if leggibile != rel.name else ""
    print("   " * (len(rel.parts) - 1) + ("+ " if f.is_dir() else "- ") + rel.name + nota)

indice = next(BASE.rglob("MEMORY.md"))
print(f"\n--- {indice} (l'indice) ---")
print(indice.read_text())

topic = sorted(BASE.rglob("topics/*.md"))[0]
print(f"--- {topic} (un argomento) ---")
print(topic.read_text())


albero su disco:

+ bWVtb3J5   <- "memory"
   + dmVyb25pY2E   <- "veronica"
      + memory
         - MEMORY.md
         - state.json
         + topics
            - christopher-nolan.md
            - film-preferences.md
            - horror-lenti.md
         + transcripts
            - ab106b6e-88c4-4a32-9c91-455552217da3.jsonl

--- mem_harness/bWVtb3J5/dmVyb25pY2E/memory/MEMORY.md (l'indice) ---
# MEMORY

- [Christopher Nolan](topics/christopher-nolan.md): L'utente ama Christopher Nolan.
- [film_preferences](topics/film-preferences.md): L'utente ama Christopher Nolan e non sopporta gli horror lenti.
- [horror lenti](topics/horror-lenti.md): L'utente non sopporta gli horror lenti.

--- mem_harness/bWVtb3J5/dmVyb25pY2E/memory/topics/christopher-nolan.md (un argomento) ---
# Christopher Nolan

Updated: 2026-08-18T15:26:55+00:00
Sessions: ab106b6e-88c4-4a32-9c91-455552217da3

## Summary
L'utente ama Christopher Nolan.

## Memories
- L'utente ama Christopher Nolan.



> **Due cose che si vedono dall'output.** La prima e' **dove** scrive: `base_path / <source_id> / <utente> / memory/`, con i due segmenti di mezzo in **base64** - cosi' qualunque `user_id` (spazi, accenti, una email) diventa un nome di cartella valido, e non puo' evadere da `base_path`. Dentro trovi `MEMORY.md` (l'indice), `topics/*.md` (un file per argomento), `transcripts/` (le sessioni grezze), `state.json` (quando ha consolidato l'ultima volta). La seconda: l'estrazione automatica e' **generosa** - da **una** frase sono nati **tre** topic che si sovrappongono (`film_preferences` ridice quello che gli altri due dicono gia'). I nomi cambiano a ogni run, li decide un LLM: quello che resta costante e' la ridondanza. E' il prezzo dell'automatico, ed e' anche il motivo per cui c'e' scritto *experimental*.

### 5d. Dove va: la memoria vettoriale

La nostra `ProfiloGusti` inietta il profilo **intero**; l'harness di 5c fa un passo in piu' e pesca i topic per **parole chiave**. Entrambi si fermano prima del richiamo *per significato*.

Con una memoria che cresce non puoi piu' iniettare tutto: non la porti dentro per intero, ne **recuperi i pezzi che c'entrano**. Servono **embeddings + un vector DB**, che ritrovano i ricordi *semanticamente* vicini a quello che stai dicendo - anche quando non usano le stesse parole.

La cosa bella: **e' la stessa interfaccia**. Il `ContextProvider` che hai scritto a mano e' lo stesso punto d'aggancio dei provider di produzione - cambia una riga:

```python
# --- illustrativo, NON eseguito ---

# la nostra memoria, fatta a mano: profilo su file, iniettato per intero
Agent(client=client, instructions=FAN,
      context_providers=[ProfiloGusti(PROFILO)])

# la stessa riga, con memoria vettoriale di produzione:
from agent_framework.mem0 import Mem0ContextProvider   # pip install agent-framework-mem0
Agent(client=client, instructions=FAN,
      context_providers=[Mem0ContextProvider(source_id="gusti", user_id="veronica", search_user_id="veronica")])
#  -> richiamo semantico, al posto del profilo iniettato per intero
```

> In Python questi provider (Mem0, Redis, Neo4j, Azure AI Search) sono ancora **Preview**; il path gia' stabile e' il RAG-as-tool (`get_file_search_tool` su un vector store). Qui ci fermiamo al confine, e lo diciamo chiaro: la memoria di EP 3 **ricorda e ragiona**, non fa ricerca semantica.

## 6. Livello 3 - quando la storia non entra piu': compaction

Sessioni e memoria fanno crescere il contesto, ma la finestra del modello e' finita: prima o poi la storia va **compattata**.

`SummarizationStrategy` fa una cosa sola, e la fa in modo leggibile: **lascia intatti gli ultimi messaggi** e **sostituisce tutti quelli vecchi con un unico riassunto**, scritto dall'LLM. Due manopole:

- `target_count` - quanti messaggi tenere per davvero (si contano i **messaggi**, non i turni);
- `threshold` - di quanto si puo' sforare prima che la compattazione scatti.

`apply_compaction(messaggi, strategy=...)` e' il modo pulito di applicarla: raggruppa e annota i messaggi, chiama la strategia, e restituisce la lista compattata. Qui 16 messaggi diventano **3**: il riassunto, piu' l'ultimo turno lasciato intero.

> E' la *migration* del bullet journal: a fine mese non ricopi tutto, porti avanti cio' che conta.

In [9]:
from agent_framework import Message, apply_compaction, SummarizationStrategy

# una lunga serata di chiacchiere: 8 turni sui film di Nolan = 16 messaggi
temi = ["Inception", "Interstellar", "Tenet", "Dunkirk", "Memento", "The Prestige", "Insomnia", "Following"]
storia = []
for t in temi:
    storia.append(Message(role="user", contents=[f"Parlami di {t}."]))
    storia.append(Message(role="assistant", contents=[f"{t} e' un film di Christopher Nolan su tempo e memoria, dalla struttura non lineare."]))

def caratteri(messaggi):
    return sum(len(m.text or "") for m in messaggi)

n_prima, car_prima = len(storia), caratteri(storia)   # da leggere PRIMA: la lista viene modificata sul posto
originali = {id(m) for m in storia}                   # per riconoscere dopo il messaggio NUOVO: il riassunto
print(f"PRIMA: {n_prima} messaggi, {car_prima} caratteri")

# target_count=2 -> tieni gli ultimi 2 messaggi, riassumi tutto il resto in UNO
strat = SummarizationStrategy(client=OpenAIChatClient(model=MODEL), target_count=2, threshold=0)
compattati = await apply_compaction(storia, strategy=strat)

print(f"DOPO : {len(compattati)} messaggi, {caratteri(compattati)} caratteri")
print("-" * 72)
for i, m in enumerate(compattati, 1):
    if id(m) in originali:
        etichetta = "tenuto intatto"
    else:
        etichetta = f"RIASSUNTO: 1 messaggio nuovo al posto di {n_prima - len(compattati) + 1}"
    print(f"\n[{i}] {m.role} - {etichetta}\n{m.text}")


PRIMA: 16 messaggi, 882 caratteri
DOPO : 3 messaggi, 632 caratteri
------------------------------------------------------------------------

[1] assistant - RIASSUNTO: 1 messaggio nuovo al posto di 14
La conversazione verte su vari film di Christopher Nolan, tra cui *Inception*, *Interstellar*, *Tenet*, *Dunkirk*, *Memento*, *The Prestige* e *Insomnia*. Ogni film è descritto come affrontante temi di tempo e memoria, con una struttura non lineare. L'utente chiede informazioni su ciascun film, e l'assistente presenta risposte coerenti per tutti. Nonostante le similitudini nelle descrizioni, i titoli dei film rimangono distintivi. La discussione rimane focalizzata sull'analisi di questi specifici lavori del regista.

[2] user - tenuto intatto
Parlami di Following.

[3] assistant - tenuto intatto
Following e' un film di Christopher Nolan su tempo e memoria, dalla struttura non lineare.


> In un agente vero la strategia si aggancia col parametro `compaction_strategy` (sul chat client / `Agent`), e MAF la applica prima delle chiamate al modello. Oltre a `SummarizationStrategy` c'e' `ContextWindowCompactionStrategy(max_context_window_tokens=..., max_output_tokens=...)`, che compatta in base al **budget di token** del modello: l'angolo piu' "production".

## 7. Bonus - il Debate Club che ti conosce

Mettiamo insieme i pezzi: il Fan discute film **diversi** in sessioni diverse, e la memoria accumula i tuoi gusti. Cosi' il verdetto e' personalizzato - e da EP 4, quando nascera' il Debate Club, i debater potranno gia' tenerne conto.

In [10]:
# nuovo profilo, per vedere l'accumulo da zero
PROFILO2 = Path("profilo_debate.txt")
if PROFILO2.exists(): PROFILO2.unlink()

async def sessione(slug: str, messaggio: str):
    """Una serata al cinema: nuova istanza del Fan che condivide lo stesso profilo su file."""
    fan = Agent(client=OpenAIChatClient(model=MODEL), name="Fan",
                instructions=FAN, context_providers=[ProfiloGusti(PROFILO2)])
    r = await fan.run(f"{messaggio}\n\nDossier:\n{dossier_str(slug)}", session=fan.create_session())
    return r.text.strip()

print("Serata 1 (gli diciamo un gusto forte):")
print(await sessione("last_airbender", "Ti avviso: adoro Christopher Nolan e detesto i film per ragazzi mal riusciti. Cosa mi dici di questo?"))

print("\nSerata 2 (film nuovo, ma ci conosce gia'):")
print(await sessione("sixth_sense", "E questo film, per me com'e'?"))

print("\n--- cosa ha imparato di te ---")
print(PROFILO2.read_text())

Serata 1 (gli diciamo un gusto forte):
"The Last Airbender" di M. Night Shyamalan è un film che, nonostante un incasso di $319.7 million, ha ricevuto una critica devastante, con un punteggio basso del 5% su Rotten Tomatoes. Roger Ebert e Metacritic evidenziano problemi significativi come la recitazione scadente e una trama deludente, che si scontrano con l'aspettativa di un'avventura avvincente. Se ami Christopher Nolan, probabilmente apprezzerai la complessità e la profondità dei suoi film molto più di questo tentativo mal riuscito di adattare un'opera amata.

Serata 2 (film nuovo, ma ci conosce gia'):
"The Sixth Sense" è un capolavoro di M. Night Shyamalan che ha incassato $672.8 milioni e ha ricevuto un punteggio impressionante dell'86% su Rotten Tomatoes. Le recensioni sono per lo più entusiastiche, lodando la narrazione brillante e la tensione crescente, perfetta per chi ama i thriller psicologici. Anche se alcune critiche menzionano l'ending come una "stretcher", il film rimane u

## 8. Il win + verso EP 4

Abbiamo dato al Fan tre tipi di memoria: **sessione** (una conversazione), **persistente** (chi sei, tra le sessioni), **compaction** (per non sforare). Ricorda te - ma i **numeri sui film** continua a prenderli da un dossier gia' pronto.

**Esercizio**: aggiungi al `ProfiloGusti` uno **storico dei dibattiti** (su quali film avete gia' discusso), e fai aprire al Fan con "l'altra volta su *Old* avevi storto il naso...".

**Teaser EP 4**: un agente che ricorda e' pronto a **parlare con altri agenti**. E gli serve qualcuno che gli porti i dati freschi: il Ricercatore. Da qui nasce il **Debate Club**.